In [ ]:
import os
import numpy as np
from sklearn.cluster import KMeans
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import shutil

In [ ]:
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

def encode_image(path):

    img = Image.open(path).convert("RGB")

    inputs = processor(
        images=img,
        return_tensors="pt"
    )

    with torch.no_grad():

        vision_outputs = clip_model.vision_model(
            pixel_values=inputs["pixel_values"]
        )

        pooled = vision_outputs.pooler_output

        feat = clip_model.visual_projection(
            pooled
        )

    feat = torch.nn.functional.normalize(
        feat,
        p=2,
        dim=-1,
    )

    feat = feat.squeeze().cpu().numpy()

    return feat.tolist()

In [ ]:
def get_image_paths(folder):
    return [os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.png', '.jpeg'))] #add ext if needed


In [ ]:
!unzip /content/dataset_for_prototype_selection.zip

In [ ]:
def select_prototypes_classwise(dataset_path, k=10):

    class_to_prototypes = {}

    for cls in os.listdir(dataset_path):

        class_folder = os.path.join(dataset_path, cls)
        if not os.path.isdir(class_folder):
            continue

        image_paths = get_image_paths(class_folder)

        if len(image_paths) <= k:
            class_to_prototypes[cls] = image_paths
            continue

        # embed images
        vectors = np.array([encode_image(p) for p in image_paths])
        print(vectors.shape)

        # cluster
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(vectors)

        centers = km.cluster_centers_

        prototypes = []

        for c in centers:
            idx = np.argmin(np.linalg.norm(vectors - c, axis=1))
            prototypes.append(image_paths[idx])

        class_to_prototypes[cls] = prototypes

    return class_to_prototypes

In [ ]:
class_prototypes = select_prototypes_classwise("/content/dataset_for_prototype_selection")
print(class_prototypes)

In [ ]:
def create_prototype_dataset(class_prototypes, output_dir="RicePrototypeDataset"):

    # remove old folder if exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    os.makedirs(output_dir, exist_ok=True)

    for cls, prototype_paths in class_prototypes.items():

        class_dir = os.path.join(output_dir, cls)
        os.makedirs(class_dir, exist_ok=True)

        for img_path in prototype_paths:
            img_name = os.path.basename(img_path)
            dst_path = os.path.join(class_dir, img_name)
            shutil.copy2(img_path, dst_path)

    print(f"✅ Prototype dataset created at: {output_dir}")

In [ ]:
create_prototype_dataset(class_prototypes)

In [ ]:
!zip -r RicePrototypeDataset.zip RicePrototypeDataset